# E-Commerce Behaviour Pipeline
## Big Data Project — Group A7

Multi-stage distributed data pipeline processing **~14.6 GB** of e-commerce event logs using **Dask** and **Apache Spark**.

| | |
|---|---|
| **Dataset** | [Kaggle — eCommerce behavior data from multi-category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) |
| **Files** | `2019-Oct.csv` (5.6 GB) · `2019-Nov.csv` (9.0 GB) |
| **Schema** | `event_time`, `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session` |

### Pipeline Stages
1. **Ingestion & Validation** — load CSVs, schema enforcement, null / range checks, cleaning
2. **Dask Processing** — `filter`, `aggregate`, `pivot`, `assign`, advanced `groupby` (5 analyses)
3. **Spark Processing** — same questions via PySpark + window function, reads **raw CSV directly**
4. **Storage** — intermediate + final results written as **Parquet**
5. **Framework Comparison** — timing and ergonomics summary

In [1]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')   # make src/ importable from notebooks/

import dask
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from pathlib import Path
import pandas as pd

from src.config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, LOGS_DIR,
    RAW_OCT, RAW_NOV,
    PARQUET_VALIDATED, RESULTS_DIR,
    VALID_EVENT_TYPES, DTYPES,
)
from src.logger     import StructuredLogger
from src.validation import validate_schema, data_quality_report, clean_data
from src.ingestion  import load_csv, load_csvs
from src.transformations_dask import (
    compute_revenue_by_category,
    compute_conversion_funnel,
    compute_hourly_activity,
    compute_session_stats,
    compute_top_brands,
)
from src.storage import save_parquet_dask, save_parquet_pandas, load_parquet

try:
    from pyspark.sql import SparkSession
    from src.transformations_spark import (
        get_spark_session, get_spark_dashboard_url,
        load_csv_spark,
        compute_revenue_by_category_spark,
        compute_conversion_funnel_spark,
        compute_window_rank_spark,
        compute_hourly_activity_spark,
    )
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    print('PySpark not found — Spark stage will be skipped.')

log = StructuredLogger('pipeline')
timings: dict[str, float] = {}

for d in [OUTPUT_DIR, LOGS_DIR, PARQUET_VALIDATED, RESULTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Spark        : {SPARK_AVAILABLE}')

Project root : C:\Users\kontr\Documents\GitHub\Group A7 Project
Data dir     : C:\Users\kontr\Documents\GitHub\Group A7 Project\data
Output dir   : C:\Users\kontr\Documents\GitHub\Group A7 Project\output
Spark        : True


---
## Stage 1 — Data Ingestion & Validation

**Operations:** load two CSVs → schema check → null / value-range assertions → row-level cleaning  
**Error handling:** `on_bad_lines='skip'` drops malformed CSV rows; retry logic handles transient I/O failures; `RuntimeError` raised on schema mismatch.

> **Note:** `load_csvs()` passes both file paths directly to `dd.read_csv` as a list — Dask handles this natively without any concat or merge pass. All computation is **lazy** until `save_parquet_dask()` triggers the first full read.

In [ ]:
log.info('pipeline_started')

# ── Load both CSVs in one lazy read (no merge/concat pass) ───────────────────
with log.timer('ingestion') as t:
    raw_ddf = load_csvs([RAW_OCT, RAW_NOV])
timings['ingestion'] = t.elapsed

print(f'Partitions : {raw_ddf.npartitions}')
print(f'Columns    : {list(raw_ddf.columns)}')

# ── Schema validation (fast — no compute triggered) ──────────────────────────
schema = validate_schema(raw_ddf)
print(f'\nSchema valid : {schema["valid"]}')
if not schema['valid']:
    raise RuntimeError(f'Missing columns: {schema["missing"]}')

raw_ddf.head(3)

In [ ]:
# ── Clean (lazy — still no compute) ──────────────────────────────────────────
with log.timer('cleaning') as t:
    clean_ddf = clean_data(raw_ddf)
timings['cleaning'] = t.elapsed

# ── Persist validated data as Parquet, partitioned by event_type ──────────────
# This is the first cell that triggers real computation:
#   read 14.6 GB CSV → filter → write Parquet. Expect 10-30 min on a laptop.
print('Writing validated Parquet — this triggers the first full read of the CSV files.')
print('Expected time: 10-30 minutes depending on disk speed.\n')
with log.timer('save_validated') as t:
    save_parquet_dask(clean_ddf, PARQUET_VALIDATED, partition_on=['event_type'])
timings['save_validated'] = t.elapsed

print(f'\nValidated Parquet → {PARQUET_VALIDATED}')
print(f'Elapsed           : {timings["save_validated"]:.1f}s')

# Reload from Parquet — all Dask analyses read this instead of the raw CSVs
validated_ddf = load_parquet(PARQUET_VALIDATED)
print(f'Reloaded partitions: {validated_ddf.npartitions}')

---
## Stage 2 — Distributed Processing with Dask

Five analyses, each using a distinct operation class:

| # | Analysis | Operations | Scheduler |
|---|---|---|---|
| 1 | Revenue by category | `filter` + `groupby` aggregate | distributed |
| 2 | Conversion funnel | per-event-type `filter` + single-key `groupby` | distributed |
| 3 | Hourly activity | per-event-type `filter` + `assign` + `groupby` | distributed |
| 4 | Session statistics | map-reduce `map_partitions` + pandas re-aggregation | **threaded** (no cluster) |
| 5 | Top brands | `filter` + `nlargest` | distributed |

> **Memory note:** Analysis 4 runs **outside** the distributed cluster because `groupby(user_session)` over millions of unique UUIDs triggers a `repartitiontofewer` step inside Dask's `.compute()` that OOMs workers. Without an active distributed client, the threaded scheduler concatenates partition results directly in the driver process.

In [ ]:
import dask

# Allow workers to spill to disk before hitting the kill threshold
dask.config.set({
    'distributed.worker.memory.target':    0.65,
    'distributed.worker.memory.spill':     0.75,
    'distributed.worker.memory.pause':     0.85,
    'distributed.worker.memory.terminate': 0.95,
})

dask_results = {}

# ── Analyses 1, 2, 3, 5 — run inside distributed cluster ────────────────────
with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='4GB') as cluster:
    with Client(cluster) as client:
        print(f'Dashboard : {client.dashboard_link}')
        print(f'Workers   : {len(client.scheduler_info()["workers"])}\n')

        # 1 — Revenue by category
        with log.timer('dask_revenue') as t:
            dask_results['revenue'] = compute_revenue_by_category(validated_ddf)
        timings['dask_revenue'] = t.elapsed
        print('--- Top 10 Categories by Revenue ---')
        print(dask_results['revenue'].head(10).to_string(index=False))

        # 2 — Conversion funnel
        with log.timer('dask_funnel') as t:
            dask_results['funnel'] = compute_conversion_funnel(validated_ddf)
        timings['dask_funnel'] = t.elapsed
        cols = [c for c in ['top_category','view','cart','purchase','purchase_rate','cart_rate']
                if c in dask_results['funnel'].columns]
        print('\n--- Conversion Funnel (top 5 by purchase rate) ---')
        print(dask_results['funnel'].sort_values('purchase_rate', ascending=False).head(5)[cols].to_string(index=False))

        # 3 — Hourly activity
        with log.timer('dask_hourly') as t:
            dask_results['hourly'] = compute_hourly_activity(validated_ddf)
        timings['dask_hourly'] = t.elapsed
        print('\n--- Events by Hour (purchases) ---')
        hourly_purchase = dask_results['hourly'][dask_results['hourly']['event_type'] == 'purchase']
        for _, row in hourly_purchase.iterrows():
            bar = '#' * (int(row['count']) // 50_000)
            print(f'  {int(row["hour"]):2d}:00  {int(row["count"]):>10,}  {bar}')

        # 5 — Top brands
        with log.timer('dask_brands') as t:
            dask_results['brands'] = compute_top_brands(validated_ddf, top_n=20)
        timings['dask_brands'] = t.elapsed
        print('\n--- Top 10 Brands by Purchase Revenue ---')
        print(dask_results['brands'].head(10).to_string(index=False))

# ── Analysis 4: Session stats — no active distributed client ─────────────────
print('\n--- Session Analysis (threaded scheduler, no distributed client) ---')
with log.timer('dask_sessions') as t:
    dask_results['sessions'] = compute_session_stats(validated_ddf)
timings['dask_sessions'] = t.elapsed
sess = dask_results['sessions']
print(f'  Total sessions      : {len(sess):,}')
print(f'  Avg events/session  : {sess["num_events"].mean():.1f}')
print(f'  Median duration     : {sess["session_duration_min"].median():.1f} min')
print(f'  Avg spend/session   : {sess["total_spend"].mean():.2f}')

print(f'\nDask compute total: {sum(v for k,v in timings.items() if k.startswith("dask_")):.1f}s')

In [ ]:
# ── Persist Dask results as Parquet ───────────────────────────────────────────
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_parquet_pandas(dask_results['revenue'],                RESULTS_DIR / 'dask_revenue.parquet')
save_parquet_pandas(dask_results['funnel'],                 RESULTS_DIR / 'dask_funnel.parquet')
save_parquet_pandas(dask_results['hourly'],                 RESULTS_DIR / 'dask_hourly.parquet')
save_parquet_pandas(dask_results['brands'],                 RESULTS_DIR / 'dask_brands.parquet')
save_parquet_pandas(dask_results['sessions'].head(500_000), RESULTS_DIR / 'dask_sessions.parquet')

print('Dask results saved to:', RESULTS_DIR)

---
## Stage 3 — Distributed Processing with Apache Spark

Same analytical questions answered via PySpark for framework comparison.  
Spark reads **directly from the raw CSV files** to avoid Parquet timestamp format incompatibilities between pyarrow (Dask) and the JVM Parquet reader (Spark).  
**Bonus**: analysis 3 uses a **window function** (`RANK() OVER PARTITION BY`) to rank products within each category.

> If PySpark is not installed, this stage is skipped automatically.

In [2]:
spark_results = {}

if not SPARK_AVAILABLE:
    print('Spark not available — skipping Stage 3.')
else:
    from pyspark.sql import functions as F

    spark = get_spark_session('EcomPipeline-A7')
    print(f'Dashboard    : {get_spark_dashboard_url(spark)}')
    print(f'Spark version: {spark.version}\n')

    # Spark reads the raw CSVs directly — avoids pyarrow/JVM Parquet incompatibility
    # Use forward-slash paths (Spark on Windows requires this)
    csv_paths = [
        str(RAW_OCT).replace('\\', '/'),
        str(RAW_NOV).replace('\\', '/'),
    ]
    with log.timer('spark_load') as t:
        sdf_raw = load_csv_spark(spark, csv_paths)
    timings['spark_load'] = t.elapsed

    # Apply the same cleaning filters as Stage 1
    valid_events = list(VALID_EVENT_TYPES)
    sdf = (
        sdf_raw
        .dropna(subset=['user_id', 'product_id', 'price', 'event_time', 'user_session'])
        .filter(F.col('event_type').isin(valid_events))
        .filter(F.col('price') >= 0)
        .filter(F.trim(F.col('user_session')) != '')
    )
    print(f'Spark partitions after cleaning: {sdf.rdd.getNumPartitions()}')
    sdf.printSchema()

    # 1 — Revenue by category
    with log.timer('spark_revenue') as t:
        spark_results['revenue'] = compute_revenue_by_category_spark(sdf)
    timings['spark_revenue'] = t.elapsed
    print('--- Spark: Top 10 Categories by Revenue ---')
    spark_results['revenue'].show(10, truncate=False)

    # 2 — Conversion funnel
    with log.timer('spark_funnel') as t:
        spark_results['funnel'] = compute_conversion_funnel_spark(sdf)
    timings['spark_funnel'] = t.elapsed
    print('--- Spark: Conversion Funnel ---')
    spark_results['funnel'].orderBy('purchase_rate', ascending=False).show(10, truncate=False)

    # 3 — Window function: rank products within category
    with log.timer('spark_window') as t:
        spark_results['window'] = compute_window_rank_spark(sdf)
    timings['spark_window'] = t.elapsed
    print('--- Spark: Top Products per Category (window rank) ---')
    spark_results['window'].show(20, truncate=False)

    # 4 — Hourly activity
    with log.timer('spark_hourly') as t:
        spark_results['hourly'] = compute_hourly_activity_spark(sdf)
    timings['spark_hourly'] = t.elapsed

    # Persist Spark results as Parquet
    results_fwd = str(RESULTS_DIR).replace('\\', '/')
    spark_results['revenue'].write.mode('overwrite').parquet(f'{results_fwd}/spark_revenue')
    spark_results['funnel'].write.mode('overwrite').parquet(f'{results_fwd}/spark_funnel')
    spark_results['window'].write.mode('overwrite').parquet(f'{results_fwd}/spark_window_rank')
    spark_results['hourly'].write.mode('overwrite').parquet(f'{results_fwd}/spark_hourly')

    print(f'\nSpark compute total: {sum(v for k,v in timings.items() if k.startswith("spark_")):.1f}s')
    spark.stop()

{"ts": "2026-03-12T13:50:36.788625+00:00", "level": "INFO", "event": "spark_load_started"}


Dashboard    : http://host.docker.internal:4040
Spark version: 3.5.0



{"ts": "2026-03-12T13:51:29.784651+00:00", "level": "INFO", "event": "spark_csv_loaded", "files": 2, "partitions": 110}
{"ts": "2026-03-12T13:51:29.785159+00:00", "level": "INFO", "event": "spark_load_completed", "elapsed_s": 53.0}
{"ts": "2026-03-12T13:51:30.024690+00:00", "level": "INFO", "event": "spark_revenue_started"}
{"ts": "2026-03-12T13:51:30.154022+00:00", "level": "INFO", "event": "spark_revenue_done"}
{"ts": "2026-03-12T13:51:30.154532+00:00", "level": "INFO", "event": "spark_revenue_completed", "elapsed_s": 0.13}


Spark partitions after cleaning: 110
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

--- Spark: Top 10 Categories by Revenue ---


{"ts": "2026-03-12T13:52:12.039674+00:00", "level": "INFO", "event": "spark_funnel_started"}


...

--- Spark: Conversion Funnel ---


{"ts": "2026-03-12T13:53:58.873420+00:00", "level": "INFO", "event": "spark_window_started"}
{"ts": "2026-03-12T13:53:59.039709+00:00", "level": "INFO", "event": "spark_window_rank_done"}
{"ts": "2026-03-12T13:53:59.040733+00:00", "level": "INFO", "event": "spark_window_completed", "elapsed_s": 0.17}


+------------+-------+--------+--------+--------------------+--------------------+
|top_category|cart   |purchase|view    |purchase_rate       |cart_rate           |
+------------+-------+--------+--------+--------------------+--------------------+
|electronics |2198451|916667  |37026582|0.024756997553811475|0.05937493771366744 |
|medicine    |1796   |654     |34738   |0.018826645172433647|0.05170130692613276 |
|stationery  |750    |325     |19323   |0.016819334471872897|0.03881384878124515 |
|appliances  |445181 |174022  |12837916|0.013555315364269403|0.034677045713650094|
|unknown     |932216 |407643  |34073918|0.011963490667553993|0.027358638357937   |
|computers   |145266 |62332   |6297977 |0.009897146337625558|0.023065501827015247|
|auto        |48229  |21339   |2157706 |0.009889669862344545|0.022351979370683495|
|construction|46727  |16500   |1759762 |0.009376267927140147|0.026553022510998645|
|kids        |23353  |11648   |1292002 |0.009015465920331393|0.018075049419428144|
|spo

{"ts": "2026-03-12T13:54:38.404877+00:00", "level": "INFO", "event": "spark_hourly_started"}
{"ts": "2026-03-12T13:54:38.474852+00:00", "level": "INFO", "event": "spark_hourly_done"}
{"ts": "2026-03-12T13:54:38.476954+00:00", "level": "INFO", "event": "spark_hourly_completed", "elapsed_s": 0.07}


+------------+----------+------------------+---------+----------------+
|top_category|product_id|product_revenue   |num_sales|rank_in_category|
+------------+----------+------------------+---------+----------------+
|accessories |28401112  |2256.9999999999995|56       |1               |
|accessories |28401058  |2007.8000000000009|20       |2               |
|accessories |28400759  |1706.2999999999995|50       |3               |
|accessories |28400912  |1330.1700000000005|31       |4               |
|accessories |28300432  |1250.9099999999996|27       |5               |
|apparel     |28716666  |7983.890000000001 |85       |1               |
|apparel     |28720716  |7981.020000000002 |101      |2               |
|apparel     |28716978  |7519.3300000000045|91       |3               |
|apparel     |54900004  |7490.339999999998 |97       |4               |
|apparel     |28703609  |6261.949999999999 |53       |5               |
|appliances  |3600661   |974933.7500000001 |3224     |1         

---
## Stage 4 — Results Summary & Framework Comparison

### Storage Strategy

| Layer | Format | Location | Why |
|---|---|---|---|
| Raw input | CSV | `data/` | Source of truth, unchanged |
| Validated intermediate | **Parquet** (partitioned by `event_type`) | `output/parquet/validated/` | Predicate pushdown speeds downstream reads |
| Analysis results | **Parquet** (single file) | `output/results/` | Compact, typed, directly loadable by BI tools |

In [3]:
# ── Timing comparison ─────────────────────────────────────────────────────────
print('=== Pipeline Stage Timings ===')
print(f'{"Stage":<25} {"Time (s)":>10}')
print('-' * 37)
for stage, elapsed in timings.items():
    print(f'{stage:<25} {elapsed:>10.1f}')
print('=' * 37)
print(f'{"TOTAL":<25} {sum(timings.values()):>10.1f}')

# ── Output files ──────────────────────────────────────────────────────────────
print('\n=== Output Files ===')
for p in sorted(Path(OUTPUT_DIR).rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1_048_576
        print(f'  {str(p.relative_to(PROJECT_ROOT)):<55} {size_mb:>7.2f} MB')

# ── Framework notes ───────────────────────────────────────────────────────────
print("""
=== Framework Comparison Notes ===
Dask
  + Familiar pandas-like API, easy iteration in notebooks
  + Native Python, no JVM overhead
  + Low-latency scheduler startup
  - High-cardinality groupby OOMs via repartitiontofewer in distributed mode;
    must run outside Client context with threaded scheduler
  - Multi-key shuffles require per-key decomposition to avoid OOM

Spark
  + Catalyst optimiser produces efficient physical plans
  + First-class window functions, complex joins
  + Graceful spill-to-disk for large shuffles
  + No format interoperability issues (reads CSV directly)
  - JVM startup adds ~30s latency
  - More verbose API; Windows path handling quirks

Verdict: Dask is faster for simple aggregations on a single node.
Spark is more robust for complex operations (windows, multi-key joins)
and scales better on managed clusters (EMR, Dataproc).
""")

=== Pipeline Stage Timings ===
Stage                       Time (s)
-------------------------------------
spark_load                      53.0
spark_revenue                    0.1
spark_funnel                    48.3
spark_window                     0.2
spark_hourly                     0.1
TOTAL                          101.6

=== Output Files ===
  output\parquet\validated\event_type=cart\part.0.parquet    0.56 MB
  output\parquet\validated\event_type=cart\part.1.parquet    0.56 MB
  output\parquet\validated\event_type=cart\part.10.parquet    0.48 MB
  output\parquet\validated\event_type=cart\part.100.parquet    1.75 MB
  output\parquet\validated\event_type=cart\part.101.parquet    1.37 MB
  output\parquet\validated\event_type=cart\part.102.parquet    1.76 MB
  output\parquet\validated\event_type=cart\part.103.parquet    1.67 MB
  output\parquet\validated\event_type=cart\part.104.parquet    1.59 MB
  output\parquet\validated\event_type=cart\part.105.parquet    1.86 MB
  output\parquet